# CNN Facial-Region Segmentation for rPPG

This notebook trains a small **U-Net-style CNN** to segment five classes in a face:

- `0` — background
- `1` — forehead
- `2` — left cheek
- `3` — right cheek
- `4` — lower lip

The intended pipeline is:

**video frames → labeled masks → CNN segmentation → facial-region masks → RGB extraction → rPPG**

> **Important:** The model needs paired images and pixel-level masks. The masks should be reviewed/corrected if they were generated automatically with MediaPipe, because pseudo-label errors will otherwise be learned by the CNN.

## Recommended dataset structure

```text
rppg_dataset/
├── images/
│   ├── frame_00001.jpg
│   ├── frame_00002.jpg
│   └── ...
└── masks/
    ├── frame_00001.png
    ├── frame_00002.png
    └── ...
```

Each mask must be a **single-channel PNG** whose pixel values are exactly `0, 1, 2, 3, 4`.


## 1. Install dependencies

Run this once. A GPU is strongly recommended for training, but the notebook can run on CPU for testing.


In [4]:
%pip install torch torchvision opencv-python numpy matplotlib pillow scikit-learn tqdm

Note: you may need to restart the kernel to use updated packages.


## 2. Imports and configuration

In [16]:
import os
import random
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

NUM_CLASSES = 5
IMAGE_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 40
LEARNING_RATE = 1e-4

DATASET_DIR = Path("rppg_dataset")
IMAGE_DIR = DATASET_DIR / "images"
MASK_DIR = DATASET_DIR / "masks"

print("Image directory:", IMAGE_DIR.resolve())
print("Mask directory:", MASK_DIR.resolve())


Device: cpu
Image directory: C:\Users\Chinmaya\Downloads\rppg_dataset\images
Mask directory: C:\Users\Chinmaya\Downloads\rppg_dataset\masks


## 3. Check the dataset

The filenames should match between `images/` and `masks/`.

For example:

`images/frame_001.jpg`

must have:

`masks/frame_001.png`


In [9]:
image_files = sorted(
    [p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]
) if IMAGE_DIR.exists() else []

pairs = []

for image_path in image_files:
    mask_path = MASK_DIR / f"{image_path.stem}.png"
    if mask_path.exists():
        pairs.append((image_path, mask_path))

print("Images found:", len(image_files))
print("Valid image/mask pairs:", len(pairs))

if len(pairs) == 0:
    print("\nNo pairs found yet.")
    print("Create the dataset folders and add matching images/masks before training.")


Images found: 0
Valid image/mask pairs: 0

No pairs found yet.
Create the dataset folders and add matching images/masks before training.


## 4. Inspect a mask

This verifies that the masks contain the expected class IDs.

Expected values:

`0, 1, 2, 3, 4`


In [17]:
if pairs:
    image_path, mask_path = pairs[0]

    image = np.array(Image.open(image_path).convert("RGB"))
    mask = np.array(Image.open(mask_path))

    print("Image shape:", image.shape)
    print("Mask shape:", mask.shape)
    print("Unique mask values:", np.unique(mask))

    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    ax[0].imshow(image)
    ax[0].set_title("Input image")
    ax[0].axis("off")

    ax[1].imshow(mask, interpolation="nearest")
    ax[1].set_title("Training mask")
    ax[1].axis("off")

    plt.show()


## 5. Dataset class

The image is resized to `224 × 224`.

The mask uses **nearest-neighbor interpolation** so class IDs do not get corrupted.


In [18]:
class FacialRegionDataset(Dataset):
    def __init__(self, pairs, image_size=224, augment=False):
        self.pairs = pairs
        self.image_size = image_size
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        image_path, mask_path = self.pairs[idx]

        image = np.array(Image.open(image_path).convert("RGB"))
        mask = np.array(Image.open(mask_path))

        # Resize image normally
        image = cv2.resize(
            image,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_LINEAR
        )

        # Resize segmentation mask with nearest neighbor
        mask = cv2.resize(
            mask,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_NEAREST
        )

        # Simple horizontal flip augmentation
        if self.augment and random.random() < 0.5:
            image = np.ascontiguousarray(image[:, ::-1])
            mask = np.ascontiguousarray(mask[:, ::-1])

            # Swap left and right cheek labels after flipping
            mask_copy = mask.copy()
            mask[mask_copy == 2] = 3
            mask[mask_copy == 3] = 2

        image = image.astype(np.float32) / 255.0

        # ImageNet-style normalization
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        image = (image - mean) / std

        image = torch.from_numpy(image).permute(2, 0, 1).float()
        mask = torch.from_numpy(mask.astype(np.int64))

        return image, mask


## 6. Split the data

For a real research experiment, **split by person**, not randomly by individual frames.

This cell provides a frame-level split only as a fallback for a prototype dataset.

For ISEF-quality experiments, create participant IDs and perform a participant-level split.


In [19]:
if len(pairs) >= 3:
    train_pairs, temp_pairs = train_test_split(
        pairs, test_size=0.30, random_state=SEED
    )
    val_pairs, test_pairs = train_test_split(
        temp_pairs, test_size=0.50, random_state=SEED
    )
else:
    train_pairs, val_pairs, test_pairs = pairs, [], []

print("Train:", len(train_pairs))
print("Validation:", len(val_pairs))
print("Test:", len(test_pairs))

train_dataset = FacialRegionDataset(train_pairs, IMAGE_SIZE, augment=True)
val_dataset = FacialRegionDataset(val_pairs, IMAGE_SIZE, augment=False)
test_dataset = FacialRegionDataset(test_pairs, IMAGE_SIZE, augment=False)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=torch.cuda.is_available()
)


Train: 0
Validation: 0
Test: 0


ValueError: num_samples should be a positive integer value, but got num_samples=0

## 7. U-Net model

This is a compact U-Net-style segmentation network.

The encoder learns spatial features and the decoder reconstructs a pixel-level segmentation map.


In [20]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()

        self.enc1 = DoubleConv(3, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.enc4 = DoubleConv(128, 256)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec4 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = DoubleConv(64, 32)

        self.output = nn.Conv2d(32, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.output(d1)


model = UNet(NUM_CLASSES).to(DEVICE)
print("Parameters:", sum(p.numel() for p in model.parameters()))


Parameters: 7763173


## 8. Dice loss

Cross-entropy handles pixel classification well, while Dice loss helps with smaller regions such as the lower lip.


In [14]:
def dice_loss(logits, targets, num_classes=5, smooth=1.0):
    probabilities = torch.softmax(logits, dim=1)

    one_hot = F.one_hot(
        targets.clamp(0, num_classes - 1),
        num_classes=num_classes
    ).permute(0, 3, 1, 2).float()

    dims = (0, 2, 3)

    intersection = (probabilities * one_hot).sum(dims)
    denominator = probabilities.sum(dims) + one_hot.sum(dims)

    dice = (2 * intersection + smooth) / (denominator + smooth)

    return 1 - dice.mean()


def segmentation_loss(logits, targets):
    ce = F.cross_entropy(logits, targets)
    dice = dice_loss(logits, targets, NUM_CLASSES)

    return ce + dice


## 9. IoU metric

Intersection-over-Union is useful for evaluating segmentation quality.

We'll report IoU separately for:

- forehead
- left cheek
- right cheek
- lower lip

rather than relying only on overall accuracy.


In [15]:
def compute_iou(logits, targets, num_classes=5):
    predictions = torch.argmax(logits, dim=1)

    ious = []

    for class_id in range(1, num_classes):
        pred = predictions == class_id
        true = targets == class_id

        intersection = (pred & true).sum().item()
        union = (pred | true).sum().item()

        if union == 0:
            iou = np.nan
        else:
            iou = intersection / union

        ious.append(iou)

    return ious


CLASS_NAMES = {
    1: "forehead",
    2: "left cheek",
    3: "right cheek",
    4: "lower lip"
}


## 10. Training functions

In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()

    total_loss = 0.0

    for images, masks in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        logits = model(images)
        loss = segmentation_loss(logits, masks)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    return total_loss / max(len(loader.dataset), 1)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    total_loss = 0.0
    all_ious = []

    for images, masks in loader:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        logits = model(images)

        loss = segmentation_loss(logits, masks)
        total_loss += loss.item() * images.size(0)

        all_ious.append(compute_iou(logits, masks, NUM_CLASSES))

    if len(loader.dataset) == 0:
        return np.nan, np.full(NUM_CLASSES - 1, np.nan)

    mean_ious = np.nanmean(np.array(all_ious), axis=0)

    return total_loss / len(loader.dataset), mean_ious


## 11. Train the CNN

The best validation model is saved as `best_facial_region_unet.pt`.


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=4
)

best_val_loss = float("inf")
history = {
    "train_loss": [],
    "val_loss": []
}

if len(train_dataset) == 0:
    print("No training data found. Add image/mask pairs first.")
else:
    for epoch in range(EPOCHS):
        train_loss = train_one_epoch(model, train_loader, optimizer)

        if len(val_dataset) > 0:
            val_loss, val_ious = evaluate(model, val_loader)
        else:
            val_loss = np.nan
            val_ious = np.full(4, np.nan)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if not np.isnan(val_loss):
            scheduler.step(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.state_dict(), "best_facial_region_unet.pt")

        print(
            f"Epoch {epoch+1:02d}/{EPOCHS} | "
            f"Train loss: {train_loss:.4f} | "
            f"Val loss: {val_loss:.4f}"
        )

        if not np.isnan(val_loss):
            print(
                "  IoU:",
                {CLASS_NAMES[i+1]: round(val_ious[i], 3)
                 for i in range(4)}
            )

    print("Training complete.")


## 12. Plot training curves

In [ ]:
if history["train_loss"]:
    plt.figure(figsize=(8, 5))
    plt.plot(history["train_loss"], label="Train")
    plt.plot(history["val_loss"], label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and validation loss")
    plt.legend()
    plt.show()


## 13. Evaluate the best model on the test set

In [ ]:
if Path("best_facial_region_unet.pt").exists():
    model.load_state_dict(
        torch.load("best_facial_region_unet.pt", map_location=DEVICE)
    )

if len(test_dataset) > 0:
    test_loss, test_ious = evaluate(model, test_loader)

    print("Test loss:", round(test_loss, 4))
    for i, iou in enumerate(test_ious, start=1):
        print(f"{CLASS_NAMES[i]:12s} IoU: {iou:.4f}")
else:
    print("No test data available.")


## 14. Visualize predictions

The predicted mask uses the same class IDs as the training mask.


In [ ]:
def denormalize(image):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    image = image.permute(1, 2, 0).cpu().numpy()
    image = image * std + mean

    return np.clip(image, 0, 1)


@torch.no_grad()
def show_predictions(dataset, n=4):
    if len(dataset) == 0:
        print("Dataset is empty.")
        return

    model.eval()

    n = min(n, len(dataset))
    indices = np.random.choice(len(dataset), n, replace=False)

    fig, axes = plt.subplots(n, 3, figsize=(12, 4*n))

    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, idx in enumerate(indices):
        image, true_mask = dataset[idx]

        logits = model(image.unsqueeze(0).to(DEVICE))
        predicted_mask = torch.argmax(logits, dim=1)[0].cpu().numpy()

        axes[row, 0].imshow(denormalize(image))
        axes[row, 0].set_title("Input")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(true_mask.numpy(), interpolation="nearest")
        axes[row, 1].set_title("Ground truth")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(predicted_mask, interpolation="nearest")
        axes[row, 2].set_title("CNN prediction")
        axes[row, 2].axis("off")

    plt.tight_layout()
    plt.show()


show_predictions(test_dataset, n=4)


## 15. Segment a new image

Set `NEW_IMAGE` to an image containing a face. The trained CNN will generate a five-class segmentation mask.


In [ ]:
NEW_IMAGE = "new_face.jpg"

def preprocess_image(path):
    image = np.array(Image.open(path).convert("RGB"))

    original = image.copy()

    resized = cv2.resize(
        image,
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=cv2.INTER_LINEAR
    )

    resized = resized.astype(np.float32) / 255.0

    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    resized = (resized - mean) / std

    tensor = torch.from_numpy(resized).permute(2, 0, 1).float()

    return original, tensor


if Path(NEW_IMAGE).exists():
    original, tensor = preprocess_image(NEW_IMAGE)

    model.eval()

    with torch.no_grad():
        logits = model(tensor.unsqueeze(0).to(DEVICE))
        prediction = torch.argmax(logits, dim=1)[0].cpu().numpy()

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.imshow(original)
    plt.title("Input")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(prediction, interpolation="nearest")
    plt.title("CNN segmentation")
    plt.axis("off")

    plt.show()
else:
    print(f"Put an image at {NEW_IMAGE} to test the model.")


## 16. Extract RGB values from each predicted region

This is the bridge from segmentation to rPPG.

For each frame, the predicted masks can be used to calculate the average R/G/B values of each facial region.


In [ ]:
def extract_region_rgb(image_rgb, segmentation_mask):
    results = {}

    for class_id, name in CLASS_NAMES.items():
        pixels = image_rgb[segmentation_mask == class_id]

        if len(pixels) == 0:
            results[name] = np.array([np.nan, np.nan, np.nan])
        else:
            results[name] = pixels.mean(axis=0)

    return results


if Path(NEW_IMAGE).exists():
    rgb_values = extract_region_rgb(
        original,
        cv2.resize(
            prediction.astype(np.uint8),
            (original.shape[1], original.shape[0]),
            interpolation=cv2.INTER_NEAREST
        )
    )

    for region, values in rgb_values.items():
        print(region, "RGB =", np.round(values, 2))


# Next step: video inference

Once this image segmentation model works, the next stage is to process a complete video:

```text
video
  ↓
frame 1 → CNN → forehead/cheeks/lower lip masks → RGB
frame 2 → CNN → forehead/cheeks/lower lip masks → RGB
frame 3 → CNN → forehead/cheeks/lower lip masks → RGB
...
  ↓
time-series RGB signals
  ↓
POS / CHROM
  ↓
pulse waveform
  ↓
heart rate
```

For an ISEF project, keep the **test participants completely separate from training participants**. Also report per-region IoU and rPPG heart-rate error rather than only reporting overall segmentation accuracy.
